In [1]:
#Importing Required Libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA

In [2]:
#Importing Required Libraries
from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet
)

from sklearn.ensemble import  RandomForestRegressor

from sklearn.metrics import r2_score,mean_squared_error

from xgboost import XGBRegressor


In [3]:
## As the Training Data and Testing Data is seperately available directly reading the data from available csv files
# No need of train_test_split function over here.
# Loading Data from Csv
train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")

In [4]:
# Seperating Target from features from train data and saving the test IDs
y = train["y"]
test_id = test["ID"]

In [5]:
#Dropping of ID columns in train and test data
train.drop(["ID", "y"], axis=1, inplace=True)
test.drop(["ID"], axis=1, inplace=True)

In [6]:
train.head()

,X0,X1,X2,X3,X4,X5,X6,X8,X10,X11,...,X375,X376,X377,X378,X379,X380,X382,X383,X384,X385
0,k,v,at,a,d,u,j,o,0,0,...,0,0,1,0,0,0,0,0,0,0
1,k,t,av,e,d,y,l,o,0,0,...,1,0,0,0,0,0,0,0,0,0
2,az,w,n,c,d,x,j,x,0,0,...,0,0,0,0,0,0,1,0,0,0
3,az,t,n,f,d,x,l,e,0,0,...,0,0,0,0,0,0,0,0,0,0
4,az,v,n,f,d,h,d,n,0,0,...,0,0,0,0,0,0,0,0,0,0


In [7]:
test.head()

,X0,X1,X2,X3,X4,X5,X6,X8,X10,X11,...,X375,X376,X377,X378,X379,X380,X382,X383,X384,X385
0,az,v,n,f,d,t,a,w,0,0,...,0,0,0,1,0,0,0,0,0,0
1,t,b,ai,a,d,b,g,y,0,0,...,0,0,1,0,0,0,0,0,0,0
2,az,v,as,f,d,a,j,j,0,0,...,0,0,0,1,0,0,0,0,0,0
3,az,l,n,f,d,z,l,n,0,0,...,0,0,0,1,0,0,0,0,0,0
4,w,s,as,c,d,y,i,m,0,0,...,1,0,0,0,0,0,0,0,0,0


In [8]:
train.describe()

,X10,X11,X12,X13,X14,X15,X16,X17,X18,X19,...,X375,X376,X377,X378,X379,X380,X382,X383,X384,X385
count,4209.000000,4209.0,4209.000000,4209.000000,4209.000000,4209.000000,4209.000000,4209.000000,4209.000000,4209.000000,...,4209.000000,4209.000000,4209.000000,4209.000000,4209.000000,4209.000000,4209.000000,4209.000000,4209.000000,4209.000000
mean,0.013305,0.0,0.075077,0.057971,0.428130,0.000475,0.002613,0.007603,0.007840,0.099549,...,0.318841,0.057258,0.314802,0.020670,0.009503,0.008078,0.007603,0.001663,0.000475,0.001426
std,0.114590,0.0,0.263547,0.233716,0.494867,0.021796,0.051061,0.086872,0.088208,0.299433,...,0.466082,0.232363,0.464492,0.142294,0.097033,0.089524,0.086872,0.040752,0.021796,0.037734
min,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.0,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,1.000000,0.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [9]:
test.describe()

,X10,X11,X12,X13,X14,X15,X16,X17,X18,X19,...,X375,X376,X377,X378,X379,X380,X382,X383,X384,X385
count,4209.000000,4209.000000,4209.000000,4209.000000,4209.000000,4209.000000,4209.000000,4209.000000,4209.000000,4209.000000,...,4209.000000,4209.000000,4209.000000,4209.000000,4209.000000,4209.000000,4209.000000,4209.000000,4209.000000,4209.000000
mean,0.019007,0.000238,0.074364,0.061060,0.427893,0.000713,0.002613,0.008791,0.010216,0.111665,...,0.325968,0.049656,0.311951,0.019244,0.011879,0.008078,0.008791,0.000475,0.000713,0.001663
std,0.136565,0.015414,0.262394,0.239468,0.494832,0.026691,0.051061,0.093357,0.100570,0.314992,...,0.468791,0.217258,0.463345,0.137399,0.108356,0.089524,0.093357,0.021796,0.026691,0.040752
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [10]:
train.shape

(4209, 376)

In [11]:
test.shape

(4209, 376)

In [12]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4209 entries, 0 to 4208
Columns: 376 entries, X0 to X385
dtypes: int64(368), object(8)
memory usage: 12.1+ MB


In [13]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4209 entries, 0 to 4208
Columns: 376 entries, X0 to X385
dtypes: int64(368), object(8)
memory usage: 12.1+ MB


In [14]:
#Checking for any missing or null values in train data
train.isnull().sum()

X0      0
X1      0
X2      0
X3      0
X4      0
       ..
X380    0
X382    0
X383    0
X384    0
X385    0
Length: 376, dtype: int64

In [15]:
#checking for any missing or null values in test data
test.isnull().sum()

X0      0
X1      0
X2      0
X3      0
X4      0
       ..
X380    0
X382    0
X383    0
X384    0
X385    0
Length: 376, dtype: int64

In [16]:
#checking for unique values in a columns
train.nunique()

X0      47
X1      27
X2      44
X3       7
X4       4
        ..
X380     2
X382     2
X383     2
X384     2
X385     2
Length: 376, dtype: int64

In [17]:
#checking for any unique values in columns
test.nunique()

X0      49
X1      27
X2      45
X3       7
X4       4
        ..
X380     2
X382     2
X383     2
X384     2
X385     2
Length: 376, dtype: int64

In [18]:
# Applying Label Encoding to convert Categorical Data to Numerical
for col in train.columns:

    if train[col].dtype == "object":

        le = LabelEncoder()

        vals = (
            list(train[col].astype(str))
            +
            list(test[col].astype(str))
        )

        le.fit(vals)

        train[col] = le.transform(train[col].astype(str))
        test[col]  = le.transform(test[col].astype(str))

In [19]:
# Removing columns with zero variance

combined = pd.concat([train, test], axis=0)

constant_cols = [col for col in combined.columns if combined[col].nunique() == 1]

train.drop(columns=constant_cols, inplace=True)
test.drop(columns=constant_cols, inplace=True)

print("Columns with zero variance:", len(constant_cols))

Columns with zero variance: 0


In [20]:
# Setting up the K-Fold Technique

kf = KFold( n_splits=5,shuffle=True, random_state=42)

In [21]:
# intiliazing empty list to store the scores from each iteration

# LINEAR REGRESSION SCORES

train_r2_lr_list = []
val_r2_lr_list   = []

train_rmse_lr_list = []
val_rmse_lr_list   = []


# RIDGE SCORES

train_r2_ridge_list = []
val_r2_ridge_list   = []

train_rmse_ridge_list = []
val_rmse_ridge_list   = []


# LASSO SCORES

train_r2_lasso_list = []
val_r2_lasso_list   = []
train_rmse_lasso_list = []
val_rmse_lasso_list   = []


# ELASTIC NET SCORES

train_r2_enet_list = []
val_r2_enet_list   = []

train_rmse_enet_list = []
val_rmse_enet_list   = []



# RANDOM FOREST SCORES
train_r2_rf_list = []
val_r2_rf_list   = []

train_rmse_rf_list = []
val_rmse_rf_list   = []

# XGBOOST SCORES
train_r2_xgb_list = []
val_r2_xgb_list   = []

train_rmse_xgb_list = []
val_rmse_xgb_list   = []

#Intially setting all the predictions of test data to zero

# TEST PREDICTIONS

test_pred_lr    = np.zeros(len(test))
test_pred_ridge = np.zeros(len(test))
test_pred_lasso = np.zeros(len(test))
test_pred_enet  = np.zeros(len(test))
test_pred_rf    = np.zeros(len(test))
test_pred_et    = np.zeros(len(test))
test_pred_xgb   = np.zeros(len(test))

In [22]:
fold = 1
for train_idx, val_idx in kf.split(train):

    X_train = train.iloc[train_idx]
    X_val   = train.iloc[val_idx]

    y_train = y.iloc[train_idx]
    y_val   = y.iloc[val_idx]

    
    # Scaling
    
    scaler = StandardScaler()

    X_train_s = scaler.fit_transform(X_train)
    X_val_s   = scaler.transform(X_val)
    X_test_s  = scaler.transform(test)

    
    # PCA
    
    pca = PCA(n_components=0.95)

    X_train_p = pca.fit_transform(X_train_s)
    X_val_p   = pca.transform(X_val_s)
    X_test_p  = pca.transform(X_test_s)

    
    # Appplying LINEAR REGRESSION
    
    lr = LinearRegression()

    lr.fit(X_train_p, y_train)

    train_pred_lr = lr.predict(X_train_p)
    val_pred_lr   = lr.predict(X_val_p)

    train_r2_lr = r2_score(y_train, train_pred_lr)
    val_r2_lr   = r2_score(y_val, val_pred_lr)

    train_rmse_lr = np.sqrt(
        mean_squared_error(y_train, train_pred_lr)
    )

    val_rmse_lr = np.sqrt(
        mean_squared_error(y_val, val_pred_lr)
    )

    train_r2_lr_list.append(train_r2_lr)
    val_r2_lr_list.append(val_r2_lr)

    train_rmse_lr_list.append(train_rmse_lr)
    val_rmse_lr_list.append(val_rmse_lr)

    test_pred_lr += lr.predict(X_test_p) / 5

     
    #Applying RIDGE Regularisation
    
    ridge = Ridge(alpha=10)

    ridge.fit(X_train_p, y_train)

    train_pred_ridge = ridge.predict(X_train_p)
    val_pred_ridge   = ridge.predict(X_val_p)

    train_r2_ridge = r2_score(y_train, train_pred_ridge)
    val_r2_ridge   = r2_score(y_val, val_pred_ridge)

    train_rmse_ridge = np.sqrt(
        mean_squared_error(y_train, train_pred_ridge)
    )

    val_rmse_ridge = np.sqrt(
        mean_squared_error(y_val, val_pred_ridge)
    )

    train_r2_ridge_list.append(train_r2_ridge)
    val_r2_ridge_list.append(val_r2_ridge)

    train_rmse_ridge_list.append(train_rmse_ridge)
    val_rmse_ridge_list.append(val_rmse_ridge)

    test_pred_ridge += ridge.predict(X_test_p) / 5

    
    # Applying LASSO Regularisation
    
    lasso = Lasso(alpha=0.01)

    lasso.fit(X_train_p, y_train)

    train_pred_lasso = lasso.predict(X_train_p)
    val_pred_lasso   = lasso.predict(X_val_p)

    train_r2_lasso = r2_score(y_train, train_pred_lasso)
    val_r2_lasso   = r2_score(y_val, val_pred_lasso)

    train_rmse_lasso = np.sqrt(
        mean_squared_error(y_train, train_pred_lasso)
    )

    val_rmse_lasso = np.sqrt(
        mean_squared_error(y_val, val_pred_lasso)
    )

    train_r2_lasso_list.append(train_r2_lasso)
    val_r2_lasso_list.append(val_r2_lasso)

    train_rmse_lasso_list.append(train_rmse_lasso)
    val_rmse_lasso_list.append(val_rmse_lasso)

    test_pred_lasso += lasso.predict(X_test_p) / 5

     
    # ELASTIC NET Regularisation
    
    enet = ElasticNet(alpha=0.01, l1_ratio=0.5)

    enet.fit(X_train_p, y_train)

    train_pred_enet = enet.predict(X_train_p)
    val_pred_enet   = enet.predict(X_val_p)

    train_r2_enet = r2_score(y_train, train_pred_enet)
    val_r2_enet   = r2_score(y_val, val_pred_enet)

    train_rmse_enet = np.sqrt(
        mean_squared_error(y_train, train_pred_enet)
    )

    val_rmse_enet = np.sqrt(
        mean_squared_error(y_val, val_pred_enet)
    )

    train_r2_enet_list.append(train_r2_enet)
    val_r2_enet_list.append(val_r2_enet)

    train_rmse_enet_list.append(train_rmse_enet)
    val_rmse_enet_list.append(val_rmse_enet)

    test_pred_enet += enet.predict(X_test_p) / 5

     
    #Applying RANDOM FOREST(Ensemble)
    
    rf = RandomForestRegressor(
        n_estimators=200,
        max_depth=8,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(X_train_p, y_train)

    train_pred_rf = rf.predict(X_train_p)
    val_pred_rf   = rf.predict(X_val_p)

    train_r2_rf = r2_score(y_train, train_pred_rf)
    val_r2_rf   = r2_score(y_val, val_pred_rf)

    train_rmse_rf = np.sqrt(
        mean_squared_error(y_train, train_pred_rf)
    )

    val_rmse_rf = np.sqrt(
        mean_squared_error(y_val, val_pred_rf)
    )

    train_r2_rf_list.append(train_r2_rf)
    val_r2_rf_list.append(val_r2_rf)
    train_rmse_rf_list.append(train_rmse_rf)
    val_rmse_rf_list.append(val_rmse_rf)

    test_pred_rf += rf.predict(X_test_p) / 5

    
    # Applying XGBOOST(Ensemble)
    
    xgb = XGBRegressor(
        n_estimators=300,
        learning_rate=0.03,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=2,
        reg_lambda=5,
        objective="reg:squarederror",
        random_state=42
    )

    xgb.fit(X_train_p, y_train)

    train_pred_xgb = xgb.predict(X_train_p)
    val_pred_xgb   = xgb.predict(X_val_p)

    train_r2_xgb = r2_score(y_train, train_pred_xgb)
    val_r2_xgb   = r2_score(y_val, val_pred_xgb)

    train_rmse_xgb = np.sqrt(
        mean_squared_error(y_train, train_pred_xgb)
    )
    val_rmse_xgb = np.sqrt(
        mean_squared_error(y_val, val_pred_xgb)
    )

    train_r2_xgb_list.append(train_r2_xgb)
    val_r2_xgb_list.append(val_r2_xgb)

    train_rmse_xgb_list.append(train_rmse_xgb)
    val_rmse_xgb_list.append(val_rmse_xgb)

    test_pred_xgb += xgb.predict(X_test_p) / 5

    print(f"\nFold {fold} completed")           # CROSS VALIDATION METRICS FOR EACH MODEL
    print("\n---- Linear Regression ----")
    print("Train R2 :", round(train_r2_lr,4))
    print("Val   R2 :", round(val_r2_lr,4))
    print("Train RMSE :", round(train_rmse_lr,4))
    print("Val   RMSE :", round(val_rmse_lr,4))

    print("\n---- Ridge Regression ----")
    print("Train R2 :", round(train_r2_ridge,4))
    print("Val   R2 :", round(val_r2_ridge,4))
    print("Train RMSE :", round(train_rmse_ridge,4))
    print("Val   RMSE :", round(val_rmse_ridge,4))

    print("\n----Lasso Regression-----")
    print("Train R2 :", round(train_r2_lasso,4))
    print("Val   R2 :", round(val_r2_lasso,4))
    print("Train RMSE :", round(train_rmse_lasso,4))
    print("Val   RMSE :", round(val_rmse_lasso,4))

    print("\n---- ElasticNet Regression ----")
    print("Train R2 :", round(train_r2_enet,4))
    print("Val   R2 :", round(val_r2_enet,4))
    print("Train RMSE :", round(train_rmse_enet,4))
    print("Val   RMSE :", round(val_rmse_enet,4))

    print("\n---- Random Forest ----")
    print("Train R2 :", round(train_r2_rf,4))
    print("Val   R2 :", round(val_r2_rf,4))
    print("Train RMSE :", round(train_rmse_rf,4))
    print("Val   RMSE :", round(val_rmse_rf,4))

    print("\n---- XGBoost ----")
    print("Train R2 :", round(train_r2_xgb,4))
    print("Val   R2 :", round(val_r2_xgb,4))
    print("Train RMSE :", round(train_rmse_xgb,4))
    print("Val   RMSE :", round(val_rmse_xgb,4))
    

    fold += 1
    
    



Fold 1 completed

---- Linear Regression ----
Train R2 : 0.5653
Val   R2 : 0.5581
Train RMSE : 8.3912
Val   RMSE : 8.2936

---- Ridge Regression ----
Train R2 : 0.5653
Val   R2 : 0.5582
Train RMSE : 8.3912
Val   RMSE : 8.2927

----Lasso Regression-----
Train R2 : 0.5653
Val   R2 : 0.5587
Train RMSE : 8.392
Val   RMSE : 8.2881

---- ElasticNet Regression ----
Train R2 : 0.5653
Val   R2 : 0.5586
Train RMSE : 8.3915
Val   RMSE : 8.2892

---- Random Forest ----
Train R2 : 0.7015
Val   R2 : 0.4751
Train RMSE : 6.9534
Val   RMSE : 9.0386

---- XGBoost ----
Train R2 : 0.6388
Val   R2 : 0.5215
Train RMSE : 7.6495
Val   RMSE : 8.6305

Fold 2 completed

---- Linear Regression ----
Train R2 : 0.6044
Val   R2 : 0.4023
Train RMSE : 7.7944
Val   RMSE : 10.607

---- Ridge Regression ----
Train R2 : 0.6044
Val   R2 : 0.4024
Train RMSE : 7.7944
Val   RMSE : 10.6059

----Lasso Regression-----
Train R2 : 0.6043
Val   R2 : 0.4031
Train RMSE : 7.7952
Val   RMSE : 10.5998

---- ElasticNet Regression ----
T

In [23]:

# FINAL RESULTS
print("\n========== FINAL RESULTS ==========")

print("\n--- Linear Regression ---")
print("Avg Train R2 :", round(np.mean(train_r2_lr_list),4))
print("Avg Val R2 :", round(np.mean(val_r2_lr_list),4))

print("\n--- Ridge ---")
print("Avg Train R2 :", round(np.mean(train_r2_ridge_list),4))
print("Avg Val R2 :", round(np.mean(val_r2_ridge_list),4))


print("\n--- Lasso ---")
print("Avg Train R2 :", round(np.mean(train_r2_lasso_list),4))
print("Avg Val R2 :", round(np.mean(val_r2_lasso_list),4))

print("\n--- ElasticNet ---")
print("Avg Train R2 :", round(np.mean(train_r2_enet_list),4))
print("Avg Val R2 :", round(np.mean(val_r2_enet_list),4))

print("\n--- Random Forest ---")
print("Avg Train R2 :", round(np.mean(train_r2_rf_list),4))
print("Avg Val R2 :", round(np.mean(val_r2_rf_list),4))

print("\n--- XGBoost ---")
print("Avg Train R2 :", round(np.mean(train_r2_xgb_list),4))
print("Avg Val R2 :", round(np.mean(val_r2_xgb_list),4))


========== FINAL RESULTS ==========

--- Linear Regression ---
Avg Train R2 : 0.5721
Avg Val R2 : 0.5269

--- Ridge ---
Avg Train R2 : 0.5721
Avg Val R2 : 0.527

--- Lasso ---
Avg Train R2 : 0.572
Avg Val R2 : 0.5275

--- ElasticNet ---
Avg Train R2 : 0.5721
Avg Val R2 : 0.5274

--- Random Forest ---
Avg Train R2 : 0.6999
Avg Val R2 : 0.4751

--- XGBoost ---
Avg Train R2 : 0.6418
Avg Val R2 : 0.5028


Even after performing of Ensembling Techniques the data is prone to Overfitting. We can see that Linear Regression gives more generalisation of the data.

Hence using LinearRegression model to predict the values of test data

In [24]:
# Submission
submission = pd.DataFrame({
    "ID": test_id,
    "y": test_pred_lr
})

submission.to_csv("final_xgboost_submission.csv", index=False)

print("\nfinal_xgboost_submission.csv created!")
print(submission.head())


final_xgboost_submission.csv created!
   ID           y
0   1  104.744138
1   2  120.627046
2   3  106.642109
3   4   72.520503
4   5  107.706897
